# Exp 5 – Erklärbarkeit: ExplainerPFN (airbnb)
- Zero-Shot Feature-Attributionen über TabPFN **2.1.2** → separater Kernel `venv_explainerpfn`
- Erklärt die **Modellvorhersagen** des TabPFN-Klassifikators (`tabpfn_preds.csv`), nicht die Labels (laut Paper)
- Gleiche 8 Features / Kontext; **StandardScaler** auf den Features; MLflow → airbnb_paris_experiment_5
- Voraussetzung: `explainers.ipynb` zuerst (erzeugt `ref_kernelshap.csv` + `tabpfn_preds.csv`)

In [13]:
import os
import sys
import time
import numpy as np
import pandas as pd
import mlflow
from scipy.stats import spearmanr, pearsonr
from sklearn.preprocessing import StandardScaler

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"  # mlflow 3.x: ./mlruns File-Store erlauben

# CWD sicher auf den exp5-Ordner setzen (Kernel-Start-CWD kann abweichen):
# Repo-Root suchen, dann in exp5 wechseln -> relative Pfade + Import funktionieren, keine .env-Kollision
while not os.path.isdir("airbnb_notebooks") and os.getcwd() != "/":
    os.chdir("..")
os.chdir("airbnb_notebooks/exp5")

sys.path.insert(0, "../../ExplainerPFN")  # geklontes Repo (keine Build-Config)
from explainerpfn.base import ExplainerPFN

## Daten & Subset
- Identische Feature-/Subset-Konstruktion wie im `explainers.ipynb`
- StandardScaler auf Kontext gefittet; `y_pred` = TabPFN-Vorhersagen aus `tabpfn_preds.csv`

In [14]:
LABEL = "is_top_rating"
TEXT_COLS = ['name', 'description', 'neighborhood_overview', 'host_about']
FEAT8 = ["calculated_host_listings_count", "estimated_occupancy_l365d", "minimum_nights", "instant_bookable", "host_tenure_days", "host_is_superhost", "calculated_host_listings_count_entire_homes", "availability_365"]  # Top-8 nach KernelSHAP-Wichtigkeit (Full-Feature-Lauf)

df = pd.read_csv("../../data/preprocessed/cleaned_text_airbnb_paris.csv", keep_default_na=False).set_index("row_id")
y = df[LABEL].astype(int)
Xnum = df[FEAT8]                # m = 8 Features (gemeinsame Obergrenze beider PFN-Explainer)
num_cols = FEAT8
outlier_label = y.value_counts().idxmin()

rng = np.random.RandomState(42)
out_idx = y.index[y == outlier_label].to_numpy()
in_idx = y.index[y != outlier_label].to_numpy()
rng.shuffle(out_idx)
rng.shuffle(in_idx)
ctx_id = np.concatenate([out_idx[:40], in_idx[:160]])           # n = 200 Kontext, 1:4
explain_id = np.concatenate([out_idx[40:50], in_idx[160:180]])  # 10 + 20 = 30 Erklär-Zeilen
outlier_col = sorted(np.unique(y.loc[ctx_id]).tolist()).index(outlier_label)
print("Kontext:", len(ctx_id), "| erklärt:", len(explain_id), "| Features:", Xnum.shape[1])

scaler = StandardScaler().fit(Xnum.loc[ctx_id])
Xs = pd.DataFrame(scaler.transform(Xnum), index=Xnum.index, columns=num_cols)
y_pred = pd.read_csv("tabpfn_preds.csv").set_index("row_id")["y_pred"]  # Modellvorhersagen (kein Label)

Kontext: 200 | erklärt: 30 | Features: 8


## ExplainerPFN
- `fit`/`predict` auf standardisierten Features, Ziel = **Modellvorhersagen** `y_pred`
- `apply_correction` (laut Paper empfohlen)

In [15]:
explainer = ExplainerPFN(n_estimators=1, device="auto", fit_mode="fit_with_cache")
explainer.fit(Xs.loc[ctx_id].values, y_pred.loc[ctx_id].values)

t0 = time.time()
expl = explainer.predict(Xs.loc[explain_id].values, y_pred.loc[explain_id].values)
corrected = explainer.apply_correction(y_pred.loc[explain_id].values, np.asarray(expl), kind=["statistical", "additive"])
t_epfn = time.time() - t0
attr = np.asarray(corrected).reshape(len(explain_id), len(num_cols))

imp = pd.Series(np.abs(attr).mean(0), index=num_cols)
display(imp.sort_values(ascending=False).round(5).to_frame("mean_abs_attr"))

,mean_abs_attr
host_tenure_days,0.12780
instant_bookable,0.11986
availability_365,0.10605
host_is_superhost,0.07183
estimated_occupancy_l365d,0.07168
minimum_nights,0.05477
calculated_host_listings_count_entire_homes,0.04988
calculated_host_listings_count,0.04116


## Fidelity vs. KernelSHAP
- Referenzmatrix laden; Pearson + Spearman (signiert) + Cosine (|attr|) je Zeile

In [16]:
ref = pd.read_csv("ref_kernelshap.csv").set_index("row_id").loc[explain_id, num_cols].values

pear = np.mean([pearsonr(ref[i], attr[i]).statistic for i in range(len(explain_id))])
spear = np.mean([spearmanr(ref[i], attr[i]).statistic for i in range(len(explain_id))])
cosine = np.mean([np.abs(ref[i]) @ np.abs(attr[i]) /
                  (np.linalg.norm(ref[i]) * np.linalg.norm(attr[i]) + 1e-12)
                  for i in range(len(explain_id))])
print(f"ExplainerPFN – t={t_epfn:.1f}s")
print(f"Fidelity vs KernelSHAP: pearson={pear:.4f}  spearman={spear:.4f}  cosine={cosine:.4f}")

mlflow.set_tracking_uri("file:../../mlruns")
mlflow.set_experiment("airbnb_paris_experiment_5")
with mlflow.start_run(run_name="explainerpfn"):
    for c in num_cols:
        mlflow.log_metric(f"num_{c}", float(imp[c]))
    mlflow.log_metric("runtime_s", round(t_epfn, 2))
    mlflow.log_metric("pearson_vs_kernelshap", float(pear))
    mlflow.log_metric("spearman_vs_kernelshap", float(spear))
    mlflow.log_metric("cosine_vs_kernelshap", float(cosine))

ExplainerPFN – t=0.5s
Fidelity vs KernelSHAP: pearson=0.2215  spearman=0.1944  cosine=0.7018
